In [43]:
import torch # type: ignore
import torch.nn as nn # type: ignore
from torch import optim # type: ignore
import torch.nn.functional as f # type: ignore
import torch_directml # type: ignore
import csv # type: ignore
import random # type: ignore
import re # type: ignore
import os # type: ignore
import unicodedata # type: ignore
import codecs # type: ignore
import itertools # type: ignore

In [44]:
# Device
# device = torch_directml.device(0)
device = 'cpu'
print(torch_directml.device_name(0))

AMD Radeon RX 6800S 


Data Preprocessing

In [45]:
# Get data pathes
lines_filepath = os.path.join('./source/cornell_movie_dialogs_corpus/cornell_movie_dialogs_corpus',
                              'movie_lines.txt')
conv_filepath = os.path.join('./source/cornell_movie_dialogs_corpus/cornell_movie_dialogs_corpus',
                             'movie_conversations.txt')

In [46]:
# Visualize data
with open(lines_filepath, 'r', encoding='iso-8859-1') as file:
    lines = file.readlines()
for line in lines[:8]:
    print(line.strip())

L1045 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ They do not!
L1044 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ They do to!
L985 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ I hope so.
L984 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ She okay?
L925 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Let's go.
L924 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ Wow
L872 +++$+++ u0 +++$+++ m0 +++$+++ BIANCA +++$+++ Okay -- you're gonna need to learn how to lie.
L871 +++$+++ u2 +++$+++ m0 +++$+++ CAMERON +++$+++ No


In [47]:
# Split each line and extract fields
line_fields = ['LineID', 'CharacterID', 'MovieID', 'Character', 'Text']
lines = {}
with open(lines_filepath, 'r', encoding='iso-8859-1') as file:
    for line in file:
        values = line.split(' +++$+++ ')
        line_object = {}
        for i, field in enumerate(line_fields):
            line_object[field] = values[i]
        lines[line_object['LineID']] = line_object

In [48]:
# Visualize data
list(lines.items())[0]

('L1045',
 {'LineID': 'L1045',
  'CharacterID': 'u0',
  'MovieID': 'm0',
  'Character': 'BIANCA',
  'Text': 'They do not!\n'})

In [49]:
# Same processing of the second file
conv_fields = ['Character1ID', 'Character2ID', 'MovieID', 'UtteranceIDs']
conversations = []
with open(conv_filepath, 'r', encoding='iso-8859-1') as file:
    for line in file:
        values = line.split(' +++$+++ ')
        conv_object = {}
        for i, field in enumerate(conv_fields):
            conv_object[field] = values[i]
        lines_ids = eval(conv_object['UtteranceIDs'])
        conv_object['Lines'] = []
        for line_id in lines_ids:
            conv_object['Lines'].append(lines[line_id])
        conversations.append(conv_object)

In [50]:
# Visualize data
conversations[0]

{'Character1ID': 'u0',
 'Character2ID': 'u2',
 'MovieID': 'm0',
 'UtteranceIDs': "['L194', 'L195', 'L196', 'L197']\n",
 'Lines': [{'LineID': 'L194',
   'CharacterID': 'u0',
   'MovieID': 'm0',
   'Character': 'BIANCA',
   'Text': 'Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.\n'},
  {'LineID': 'L195',
   'CharacterID': 'u2',
   'MovieID': 'm0',
   'Character': 'CAMERON',
   'Text': "Well, I thought we'd start with pronunciation, if that's okay with you.\n"},
  {'LineID': 'L196',
   'CharacterID': 'u0',
   'MovieID': 'm0',
   'Character': 'BIANCA',
   'Text': 'Not the hacking and gagging and spitting part.  Please.\n'},
  {'LineID': 'L197',
   'CharacterID': 'u2',
   'MovieID': 'm0',
   'Character': 'CAMERON',
   'Text': "Okay... then how 'bout we try out some French cuisine.  Saturday?  Night?\n"}]}

In [51]:
# Extract quesion - answer (pairs of dialogs) pairs
qa_pairs = []
for converstion in conversations:
    for i in range(len(converstion['Lines']) - 1):
        input_line = converstion['Lines'][i]['Text'].strip()
        target_line = converstion['Lines'][i + 1]['Text'].strip()
        if input_line and target_line:
            qa_pairs.append([input_line, target_line])

In [52]:
# Visualizr data
qa_pairs[0]

['Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.',
 "Well, I thought we'd start with pronunciation, if that's okay with you."]

In [53]:
# Save formatted data
data_filepath = os.path.join('./source/cornell_movie_dialogs_corpus/cornell_movie_dialogs_corpus',
                             'formatted_movie_lines.txt')
delimeter = '\t'
delimeter = str(codecs.decode(delimeter, 'unicode_escape'))
with open(data_filepath, 'w', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=delimeter)
    for pair in qa_pairs:
        writer.writerow(pair)
print("End of writing")

End of writing


In [54]:
# Check saving
with open(data_filepath, 'rb') as file:
    lines = file.readlines()
for line in lines[:8]:
    print(line)

b"Can we make this quick?  Roxanne Korrine and Andrew Barrett are having an incredibly horrendous public break- up on the quad.  Again.\tWell, I thought we'd start with pronunciation, if that's okay with you.\r\r\n"
b"Well, I thought we'd start with pronunciation, if that's okay with you.\tNot the hacking and gagging and spitting part.  Please.\r\r\n"
b"Not the hacking and gagging and spitting part.  Please.\tOkay... then how 'bout we try out some French cuisine.  Saturday?  Night?\r\r\n"
b"You're asking me out.  That's so cute. What's your name again?\tForget it.\r\r\n"
b"No, no, it's my fault -- we didn't have a proper introduction ---\tCameron.\r\r\n"
b"Cameron.\tThe thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does.\r\r\n"
b"The thing is, Cameron -- I'm at the mercy of a particularly hideous breed of loser.  My sister.  I can't date until she does.\tSeems like she could get a date easy enough...\r\r\n"
b'Why?\tU

Processing words

In [55]:
# Word processor class
PAD_token = 0 # Padding for short sentences
SOS_token = 1 # Start of the sentence
EOS_token = 2 # End of the sentence

class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.index2word = {PAD_token : 'PAD', SOS_token : 'SOS', EOS_token : 'EOS'}
        self.word2count = {}
        self.num_words = 3
    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)
    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.num_words
            self.index2word[self.num_words] = word
            self.word2count[word] = 1
            self.num_words += 1
        else:
            self.word2count[word] += 1
    def trim(self, min_count):
        keep_words = []
        for key, value in self.word2count.items():
            if value >= min_count:
                keep_words.append(key)
        self.word2index = {}
        self.index2word = {PAD_token : 'PAD', SOS_token : 'SOS', EOS_token : 'EOS'}
        self.word2count = {}
        self.num_words = 3
        for word in keep_words:
            self.add_word(word)

In [56]:
# Get ASCII string
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

In [57]:
# Normalize string
def normalize_string(s):
    s = unicode_to_ascii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s) # replace [.?!] -> ' ' + [.?!]
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s) # remove all other chars
    s = re.sub(r"\s+", r" ", s).strip() # remove whitespace chars
    return s

In [58]:
# Process qa_pairs and create vocabuary
lines = open(data_filepath, 'r', encoding='utf-8').read().strip().split('\n')
pairs = [[normalize_string(s) for s in pair.split('\t')] for pair in lines]
print('Done processing')
vocabulary = Vocabulary('cornell_movie_dialogs_corpus')

Done processing


In [59]:
# Visualize data
pairs[0]

['can we make this quick ? roxanne korrine and andrew barrett are having an incredibly horrendous public break up on the quad . again .',
 'well i thought we d start with pronunciation if that s okay with you .']

In [60]:
# Filter sentences by length
MAX_LENGTH = 25
def filter_pair(p):
    return len(p[0].split()) < MAX_LENGTH and len(p[1].split()) < MAX_LENGTH

def filter_pairs(pairs):
    return [pair for pair in pairs if filter_pair(pair)]

In [61]:
# Filter pairs
pairs = [pair for pair in pairs if len(pair) > 1]
pairs = filter_pairs(pairs)

In [62]:
# Visualize data
len(pairs)

168807

In [63]:
# Filling the vocabulary
for pair in pairs:
    vocabulary.add_sentence(pair[0])
    vocabulary.add_sentence(pair[1])

In [64]:
# Visualize data
vocabulary.num_words

37221

In [65]:
# Remove rare wods
MIN_COUNT = 3
def trim_rare_words(voc, pairs, MIN_COUNT):
    voc.trim(MIN_COUNT)
    keep_pairs = []
    for pair in pairs:
        input_sentence = pair[0]
        output_sentence = pair[1]
        keep_input = True
        keep_output = True
        for word in input_sentence.split(' '):
            if word not in voc.word2index:
                keep_input = False
                break
        for word in output_sentence.split(' '):
            if word not in voc.word2index:
                keep_output = False
                break
        if keep_input and keep_output:
            keep_pairs.append(pair)
    return keep_pairs

In [66]:
# Trim vocabulary
pairs = trim_rare_words(vocabulary, pairs, MIN_COUNT)

In [67]:
# Visualize data
len(pairs)

148594

Prepare data

In [68]:
# Get indexed sentence
def indexes_from_sentence(vocabulary, sentence):
    return [vocabulary.word2index[word] for word in sentence.split(' ')] + [EOS_token]

In [69]:
# Get zero padding for the sentence (Correct results only if sorted)
def zero_padding(sentences, fill_value=0):
    return list(itertools.zip_longest(*sentences, fillvalue=fill_value)) # type: ignore

In [70]:
# Get binary matrix
def binary_matrix(sentences):
    matrix = []
    for i, sequence in enumerate(sentences):
        matrix.append([])
        for token in sequence:
            if token == PAD_token:
                matrix[i].append(0)
            else:
                matrix[i].append(1)
    return matrix

In [71]:
# Get padded sentences and lengths for inputs
def get_input_variable(sentences, vocabulary):
    indexed_batch = [indexes_from_sentence(vocabulary, sentence) for sentence in sentences]
    lengths = torch.tensor([len(sentence) for sentence in indexed_batch])
    padded_list = zero_padding(indexed_batch)
    padded_variable = torch.LongTensor(padded_list)
    return padded_variable, lengths

In [72]:
# Get padded sentences, mask and max length for targets
def get_output_variable(sentences, vocabulary):
    indexed_batch = [indexes_from_sentence(vocabulary, sentence) for sentence in sentences]
    max_target_len = max([len(sentence) for sentence in indexed_batch])
    padded_list = zero_padding(indexed_batch)
    mask = binary_matrix(padded_list)
    mask = torch.BoolTensor(mask)
    padded_variable = torch.LongTensor(padded_list)
    return padded_variable, mask, max_target_len

In [73]:
# Get all information for a batch of pairs
def batch_to_train_data(vocabulary, pairs_batch):
    pairs_batch.sort(key=lambda x: len(x[0].split(' ')), reverse=True)
    input_batch = []
    output_batch = []
    for pair in pairs_batch:
        input_batch.append(pair[0])
        output_batch.append(pair[1])
    inp, lengths = get_input_variable(input_batch, vocabulary)
    out, mask, max_target_len = get_output_variable(output_batch, vocabulary)
    return inp, lengths, out, mask, max_target_len

In [74]:
# Visualize data
small_batch_size = 5
batch = [random.choice(pairs) for _ in range(small_batch_size)]
input_variable, lengths, target_variable, mask, max_target_len = batch_to_train_data(vocabulary,
                                                                                      batch)
print(f"input_variable:\n{input_variable}")
print()
print(f"lengths:\n{lengths}")
print()
print(f"target_variable:\n{target_variable}")
print()
print(f"mask:\n{mask}")
print()
print(f"max_target_len:\n{max_target_len}")

input_variable:
tensor([[    4,   165,    46,    14,  1707],
        [ 4719,    14,   202,   320,    71],
        [   96,   893,  1089,    41,   210],
        [  147,   132,    12,   768,    15],
        [ 1089,  1557,   609,    33,     2],
        [   15,  1040,   610,     2,     0],
        [    4,    33,    15,     0,     0],
        [   65,     2,     2,     0,     0],
        [   16,     0,     0,     0,     0],
        [  344,     0,     0,     0,     0],
        [ 1009,     0,     0,     0,     0],
        [  617,     0,     0,     0,     0],
        [  449,     0,     0,     0,     0],
        [ 1562,     0,     0,     0,     0],
        [  125,     0,     0,     0,     0],
        [15250,     0,     0,     0,     0],
        [   96,     0,     0,     0,     0],
        [   85,     0,     0,     0,     0],
        [    4,     0,     0,     0,     0],
        [  200,     0,     0,     0,     0],
        [   50,     0,     0,     0,     0],
        [  147,     0,     0,     0,   

Creating a model

In [75]:
# Encoder class
class EncoderRNN(nn.Module):
    def __init__(self, hidden_size, embedding, n_layers=1, dropout=0):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = embedding
        self.n_layers = n_layers
        self.dropout = dropout
        self.gru = nn.GRU(hidden_size,
                          hidden_size,
                          n_layers,
                          dropout=(0 if n_layers == 1 else dropout),
                          bidirectional=True)
    def forward(self, input_sequence, input_lengths, hidden=None):
        embedded = self.embedding(input_sequence)
        packed = torch.nn.utils.rnn.pack_padded_sequence(embedded, input_lengths)
        outputs, hidden = self.gru(packed, hidden)
        outputs, _ = torch.nn.utils.rnn.pad_packed_sequence(outputs)
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, :, self.hidden_size:]
        return outputs, hidden

In [76]:
# Attention class
class Attention(nn.Module):
    def __init__(self, method, hidden_size):
        super().__init__()
        self.method = method
        self.hidden_size = hidden_size
    def dot_score(self, hidden, encoder_outputs):
        return torch.sum(hidden * encoder_outputs,
                         dim=2)
    def forward(self, hidden, encoder_outputs):
        attention_energies = self.dot_score(hidden, encoder_outputs)
        attention_energies = attention_energies.t()
        return f.softmax(attention_energies, dim=1).unsqueeze(1)

In [77]:
# Decoder class
class DecoderRNN(nn.Module):
    def __init__(self, attention_model, embedding, hidden_size, output_size, n_layers=1, dropout=0.1):
        super().__init__()
        self.attention_model = attention_model
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout
        self.embedding = embedding
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden_size,
                          hidden_size,
                          n_layers,
                          dropout=(0 if n_layers == 1 else dropout))
        self.concat = nn.Linear(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.attention = Attention(attention_model, hidden_size)
    def forward(self, input_step, last_hidden, encoder_outputs):
        embedded = self.embedding(input_step)
        embedded = self.embedding_dropout(embedded)
        rnn_output, hidden = self.gru(embedded, last_hidden)
        attention_weights = self.attention(rnn_output, encoder_outputs)
        context = attention_weights.bmm(encoder_outputs.transpose(0, 1))
        rnn_output = rnn_output.squeeze(0)
        context = context.squeeze(1)
        concat = torch.cat((rnn_output, context), dim=1)
        concat = self.concat(concat)
        concat = f.tanh(concat)
        output = self.out(concat)
        output = f.softmax(output, dim=1)
        return output, hidden

Training

In [78]:
# Loss function
def mask_nllloss(output, target, mask):
    total_number = mask.sum()
    target = target.view(-1, 1)
    gathered_tensor = torch.gather(output, 1, target)
    cross_entropy = -torch.log(gathered_tensor)
    loss = cross_entropy.masked_select(mask)
    loss = loss.mean()
    loss = loss.to(device)
    return loss, total_number.item()

In [79]:
# Training visualization
small_batch_size = 5
batches = batch_to_train_data(vocabulary, [random.choice(pairs) for _ in range(small_batch_size)])
input_variable, lengths, target_variable, mask, max_target_len = batches
print(f"Input variable:\n{input_variable.shape}")
print(f"Lengths:\n{lengths}")
print(f"Target variable:\n{target_variable.shape}")
print(f"Mask:\n{mask.shape}")
print(f"Max_target_len:\n{max_target_len}")
print()

hidden_size = 1024
encoder_n_layers = 2
decoder_n_layers = 2
dropout = 0.1
attention_model = 'dot'
embedding = nn.Embedding(vocabulary.num_words, hidden_size)

encoder = EncoderRNN(hidden_size, embedding, encoder_n_layers, dropout)
decoder = DecoderRNN(attention_model, embedding, hidden_size, vocabulary.num_words, decoder_n_layers, dropout)
encoder = encoder.to(device)
decoder = decoder.to(device)
encoder.train()
decoder.train()
encoder_optimizer = optim.Adam(encoder.parameters(), lr=0.001, weight_decay=0.0005)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=0.001, weight_decay=0.0005)
encoder.zero_grad()
decoder.zero_grad()
input_variable = input_variable.to(device)
# lengths = lengths.to(device)
target_variable = target_variable.to(device)
mask = mask.to(device)
loss = 0
print_losses = []
n_totals = 0
encoder_output, encoder_hidden = encoder(input_variable, lengths)
print(f"Encoder output:\n{encoder_output.shape}")
print(f"Encoder hidden state:\n{encoder_hidden.shape}")
print()
decoder_input = torch.LongTensor([[SOS_token for _ in range(small_batch_size)]])
decoder_input = decoder_input.to(device)
print(f"Decoder input:\n{decoder_input.shape}")
print(decoder_input)
print()
decoder_hidden = encoder_hidden[:decoder.n_layers]
print(f"Decoder hidden state:\n{decoder_hidden.shape}")
print()
print('---------------------------------------------------------------------------')
print()

for i in range(max_target_len):
    decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_output)
    print(f"Decoder output:\n{decoder_output.shape}")
    print(f"Decoder hidden state:\n{decoder_hidden.shape}")
    print()
    decoder_input = target_variable[i].view(1, -1)
    print(f"Target variable before:\n{target_variable[i]}")
    print(target_variable[i].shape)
    print(f"Target variable after:\n{decoder_input.shape}")
    print(f"Mask\n{mask[i]}")
    print(mask[i].shape)
    print()
    mask_loss, total = mask_nllloss(decoder_output, target_variable[i], mask[i])
    print(f"Mask loss:\n{mask_loss}")
    print(f"Total:\n{total}")
    print()
    loss += mask_loss
    print_losses.append(mask_loss.item() * total)
    print(f"Losses:\n{print_losses}")
    print()
    n_totals += total
    print(f"N totals:\n{n_totals}")
    print()
    returned_loss = sum(print_losses) / n_totals
    print(f"Returned loss:\n{returned_loss}")
    print()
    print('---------------------------------------------------------------------------')
    print()

Input variable:
torch.Size([23, 5])
Lengths:
tensor([23, 19, 15,  6,  3])
Target variable:
torch.Size([12, 5])
Mask:
torch.Size([12, 5])
Max_target_len:
12

Encoder output:
torch.Size([23, 5, 1024])
Encoder hidden state:
torch.Size([4, 5, 1024])

Decoder input:
torch.Size([1, 5])
tensor([[1, 1, 1, 1, 1]])

Decoder hidden state:
torch.Size([2, 5, 1024])

---------------------------------------------------------------------------

Decoder output:
torch.Size([5, 19838])
Decoder hidden state:
torch.Size([2, 5, 1024])

Target variable before:
tensor([ 699,  324,  482,   51, 6832])
torch.Size([5])
Target variable after:
torch.Size([1, 5])
Mask
tensor([True, True, True, True, True])
torch.Size([5])

Mask loss:
9.940590858459473
Total:
5

Losses:
[49.70295429229736]

N totals:
5

Returned loss:
9.940590858459473

---------------------------------------------------------------------------

Decoder output:
torch.Size([5, 19838])
Decoder hidden state:
torch.Size([2, 5, 1024])

Target variable bef

In [80]:
# Training
def train(input_variable, lengths, target_variable, mask, max_target_len, encoder, decoder,
          encoder_optimizer, decoder_optimizer, batch_size, clip):
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    input_variable = input_variable.to(device)
    lengths = lengths.to(device)
    target_variable = target_variable.to(device)
    mask = mask.to(device)
    loss = 0
    print_losses = []
    n_totals = 0
    encoder_output, encoder_hidden = encoder(input_variable, lengths)
    decoder_input = torch.LongTensor([[SOS_token for _ in range(batch_size)]]).to(device)
    decoder_hidden = encoder_hidden[:decoder.n_layers]

    teacher_forcing_ration = 0.75
    if random.random() < teacher_forcing_ration:
        for t in range(max_target_len):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_output)
            decoder_input = target_variable[t].view(1, -1)
            mask_loss, n_total = mask_nllloss(decoder_output, target_variable[t], mask[t])
            loss += mask_loss
            print_losses.append(mask_loss.item() * n_total)
            n_totals += n_total
    else:
        for t in range(max_target_len):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden, encoder_output)
            _, topi = decoder_output.topk(1)
            decoder_input = torch.LongTensor([[topi[i][0] for i in range(batch_size)]])
            decoder_input = decoder_input.to(device)
            mask_loss, n_total = mask_nllloss(decoder_output, target_variable[t], mask[t])
            loss += mask_loss
            print_losses.append(mask_loss.item() * n_total)
            n_totals += n_total
    loss.backward()
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), clip)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), clip)
    encoder_optimizer.step()
    decoder_optimizer.step()
    return sum(print_losses) / n_totals

In [81]:
# Training proces
def train_iters(vocabulary, pairs, encoder, decoder, encoder_optimizer, decoder_optimizer,
                n_iterations, batch_size, clip, load=None):
    training_batches = [batch_to_train_data(vocabulary, [random.choice(pairs) for _ in range(batch_size)])
                        for _ in range(n_iterations)]
    start_iteration = 1
    print_loss = 0
    if load:
        start_iteration = load['iteration']
    for iteration in range(start_iteration, n_iterations + 1):
        batch = training_batches[iteration - 1]
        input_variable, lengths, target_variable, mask, max_target_len = batch
        loss = train(input_variable, lengths, target_variable, mask, max_target_len, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, batch_size, clip)
        print_loss += loss
        if iteration % 10 == 0:
            print(f"Iteration: {iteration} Loss avg: {print_loss / 10}")
            print_loss = 0
            torch.save({
                'iteration' : iteration,
                'en' : encoder.state_dict(),
                'de' : decoder.state_dict(),
                'en_opt' : encoder_optimizer.state_dict(),
                'de_opt' : decoder_optimizer.state_dict(),
                'loss' : loss,
                'vocavulary' : vocabulary.__dict__,
                'embedding' : embedding.state_dict()
            }, os.path.join('./source/chat_bot.tar'))

In [82]:
train_iters(vocabulary, pairs, encoder, decoder, encoder_optimizer, decoder_optimizer, 1000, 5, 5.0)

Iteration: 10 Loss avg: 7.490731564269508
Iteration: 20 Loss avg: 5.3911870460275395
Iteration: 30 Loss avg: 5.06354576006014
Iteration: 40 Loss avg: 5.323771578878004
Iteration: 50 Loss avg: 4.747552794265211
Iteration: 60 Loss avg: 5.018821592140569
Iteration: 70 Loss avg: 4.774265506126907
Iteration: 80 Loss avg: 4.611016829597385
Iteration: 90 Loss avg: 4.940509031619994
Iteration: 100 Loss avg: 4.856780897756334
Iteration: 110 Loss avg: 4.472917042775498
Iteration: 120 Loss avg: 4.927012917596761
Iteration: 130 Loss avg: 5.291279808667306
Iteration: 140 Loss avg: 4.6954783037702414
Iteration: 150 Loss avg: 4.626358272907552
Iteration: 160 Loss avg: 5.108259175635233
Iteration: 170 Loss avg: 4.9566060492840505
Iteration: 180 Loss avg: 4.801145777155638
Iteration: 190 Loss avg: 4.753808577857937
Iteration: 200 Loss avg: 4.893941194735714
Iteration: 210 Loss avg: 4.7170603376310405
Iteration: 220 Loss avg: 4.684756069935124
Iteration: 230 Loss avg: 4.651556903245121
Iteration: 240 Lo

KeyboardInterrupt: 